# MedNorm-VI S1 - Smoke Artifact Validation and Full-Training Readiness

**Read-only. No training, no installation, no kernel restart, no GPU required.**

This notebook validates the **real** S1 smoke artifact that a fresh Colab GPU run wrote to
Drive. A screenshot is not evidence: the actual `training_manifest.json` is read, the
checkpoint SHA-256 is recomputed from the bytes on disk, and every readiness condition is
checked against recorded values.

It also prints the **immutable model revision** that full S1 training must pin.

## RUNTIME INPUTS (both required)

Set these in the configuration cell (or as environment variables). Neither has a default
that could pass off the old artifact as the corrected one.

| Input | Meaning |
| --- | --- |
| `SMOKE_ARTIFACT_DIR` | Which artifact to validate. Defaults to the **v2** rerun directory. |
| `EXPECTED_SMOKE_CHECKPOINT_SHA256` | The 64-hex digest of the run you are accepting. **Empty by default.** |

The expected hash is *not* stored in Python source, so accepting a new run never requires a
code edit. If it is missing or malformed, this notebook prints the recomputed digest and asks
you to confirm it — it never grants readiness on its own.

## ARTIFACT LIFECYCLE

```
v1  s1_mention_first_run_smoke      historical, full_training_readiness: false, IMMUTABLE
v2  s1_mention_first_run_smoke_v2   corrected rerun after the Audit 0026 alignment fix
```

A failed **global** `pip check` does not invalidate the artifact. Environment health is
scoped to the S1 dependency closure (Audit 0024); unrelated Colab conflicts (Gradio,
IPython/jedi) are carried through as diagnostics.

## 1. Configuration and repository checkout (stdlib only)

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
REPO_URL = "https://github.com/vquclinh/MedNorm-VI.git"
REPO_REF = "main"

if "MEDNORM_DRIVE_ROOT" in os.environ:
    DRIVE_ROOT = Path(os.environ["MEDNORM_DRIVE_ROOT"])
if "MEDNORM_REPO_DIR" in os.environ:
    REPO_DIR = Path(os.environ["MEDNORM_REPO_DIR"])
if "MEDNORM_REPO_URL" in os.environ:
    REPO_URL = os.environ["MEDNORM_REPO_URL"]
if "MEDNORM_REPO_REF" in os.environ:
    REPO_REF = os.environ["MEDNORM_REPO_REF"]

# ---------------------------------------------------------------------------
# RUNTIME INPUTS - edit here (or export the matching environment variables).
# ---------------------------------------------------------------------------
# Which artifact to validate. v2 = the corrected rerun; v1 is historical evidence
# and must never be silently validated in its place.
SMOKE_ARTIFACT_VERSION = os.environ.get("MEDNORM_SMOKE_ARTIFACT_VERSION", "v5")
SMOKE_ARTIFACT_DIR = Path(os.environ.get(
    "MEDNORM_SMOKE_ARTIFACT_DIR",
    str(DRIVE_ROOT / "artifacts" / f"s1_mention_first_run_smoke_{SMOKE_ARTIFACT_VERSION}")))

# The 64-hex checkpoint digest of the run you are accepting. DELIBERATELY EMPTY:
# paste the value this notebook prints, or export the environment variable. It is
# never hardcoded in Python source, so a new run needs no code change.
EXPECTED_SMOKE_CHECKPOINT_SHA256 = os.environ.get(
    "MEDNORM_EXPECTED_SMOKE_CHECKPOINT_SHA256", "")

HISTORICAL_SMOKE_ARTIFACT_DIRS = (
    DRIVE_ROOT / "artifacts" / "s1_mention_first_run_smoke",       # v1
    DRIVE_ROOT / "artifacts" / "s1_mention_first_run_smoke_v2",    # v2
    DRIVE_ROOT / "artifacts" / "s1_mention_first_run_smoke_v3",    # v3
    DRIVE_ROOT / "artifacts" / "s1_mention_first_run_smoke_v4",    # v4
)
IN_COLAB = "google.colab" in sys.modules

# No installation and no kernel restart: this notebook needs only the standard
# library plus PyYAML, which the Colab image already provides.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)],
    check=True)
RESOLVED_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO_DIR / "src"))
print(json.dumps({
    "resolved_commit": RESOLVED_COMMIT,
    "smoke_artifact_version": SMOKE_ARTIFACT_VERSION,
    "smoke_artifact_dir": str(SMOKE_ARTIFACT_DIR),
    "historical_artifact_dirs": [str(p) for p in HISTORICAL_SMOKE_ARTIFACT_DIRS],
    "expected_checkpoint_sha256_supplied": bool(EXPECTED_SMOKE_CHECKPOINT_SHA256),
    "in_colab": IN_COLAB,
}, indent=2, sort_keys=True))

## 2. Mount Drive

In [ ]:
import importlib

if IN_COLAB:
    importlib.import_module("google.colab.drive").mount("/content/drive")
assert SMOKE_ARTIFACT_DIR.is_dir(), (
    f"smoke artifact directory not found: {SMOKE_ARTIFACT_DIR}. "
    "Run MedNorm_S1_Mention_FirstRun_Smoke.ipynb first (two passes).")
if any(SMOKE_ARTIFACT_DIR.resolve() == p.resolve() for p in HISTORICAL_SMOKE_ARTIFACT_DIRS):
    print("=" * 78)
    print("WARNING: validating a HISTORICAL artifact (v1 recorded")
    print("full_training_readiness: false; v2 had 2 unalignable; v3 had 1; v4")
    print("predates the full-corpus alignment preflight).")
    print("All are kept as evidence and are expected to fail. The corrected rerun")
    print("writes to s1_mention_first_run_smoke_v5.")
    print("=" * 78)
print(sorted(str(p.relative_to(SMOKE_ARTIFACT_DIR)) for p in SMOKE_ARTIFACT_DIR.rglob("*")))

## 3. Validate the real artifact

Every condition below is checked against a recorded value, not merely for key presence.
The checkpoint hash is recomputed from the bytes on disk and must match **both** the
manifest and the hash observed on the confirmed Colab run.

In [ ]:
from mednorm_vi.training.s1_artifact_validation import (  # noqa: E402
    load_smoke_manifest,
    pinned_revision_from_outcome,
    validate_smoke_artifact,
)
from mednorm_vi.training.s1_mention_smoke import load_smoke_config  # noqa: E402

smoke_config = load_smoke_config(
    REPO_DIR / "configs" / "training" / "s1_mention_first_run_smoke.yaml")
manifest = load_smoke_manifest(SMOKE_ARTIFACT_DIR)

# The expected hash comes from the operator, never from source code.
outcome = validate_smoke_artifact(
    SMOKE_ARTIFACT_DIR,
    expected_corpus=smoke_config["corpus"],
    expected_checkpoint_sha256=EXPECTED_SMOKE_CHECKPOINT_SHA256,
)
print(json.dumps(outcome.as_dict(), indent=2, sort_keys=True))
if not outcome.diagnostics["expected_checkpoint_sha256_supplied"]:
    print("=" * 78)
    print("EXPECTED CHECKPOINT HASH NOT CONFIRMED.")
    print("Recomputed SHA-256 of the checkpoint on Drive:")
    print(f"    {outcome.checkpoint_sha256}")
    print("If that is the run you intend to accept, set it in the configuration cell:")
    print(f'    EXPECTED_SMOKE_CHECKPOINT_SHA256 = "{outcome.checkpoint_sha256}"')
    print("or export MEDNORM_EXPECTED_SMOKE_CHECKPOINT_SHA256, then re-run.")
    print("No Python source needs to change.")
    print("=" * 78)

## 4. Non-blocking environment diagnostics (preserved, never fatal)

In [ ]:
environment = manifest.get("environment", {})
print(json.dumps({
    "pip_check_passed": environment.get("pip_check_passed"),
    "s1_dependency_closure_verified": environment.get("s1_dependency_closure_verified"),
    "blocking_dependency_conflicts": environment.get("blocking_dependency_conflicts", []),
    "non_blocking_dependency_conflicts": environment.get(
        "non_blocking_dependency_conflicts", []),
}, indent=2, sort_keys=True))
print("pip check output recorded by the smoke run (complete):")
print(environment.get("pip_check_output", "") or "(none recorded)")
print()
print("A false global pip_check_passed does NOT invalidate this artifact:")
print("S1 health is scoped to the S1 dependency closure (Audit 0024).")

## 5. Verdict (fail-fast, lists every failed condition)

In [ ]:
if not outcome.smoke_validated:
    print("=" * 78)
    print("SMOKE ARTIFACT VALIDATION FAILED -", len(outcome.failures), "condition(s):")
    for failure in outcome.failures:
        print("  -", failure)
    print("=" * 78)
    print("Full-training readiness is NOT granted. Do not run full S1 training.")
    raise AssertionError(
        f"smoke artifact validation failed: {len(outcome.failures)} condition(s)")

SMOKE_ARTIFACT_VALIDATED = True
print("SMOKE ARTIFACT VALIDATED - every condition passed.")
print("checkpoint sha256:", outcome.checkpoint_sha256)

## 6. Immutable model revision for full training

Full S1 training must pin the exact commit hash resolved during the smoke run. If the
manifest carries no immutable revision, this cell **stops** and reports a blocker rather
than inventing a value.

In [ ]:
PINNED_MODEL_REVISION = pinned_revision_from_outcome(outcome)
print(json.dumps({
    "hf_model_id": manifest["model"]["hf_model_id"],
    "requested_revision": manifest["model"]["requested_revision"],
    "pinned_immutable_revision": PINNED_MODEL_REVISION,
    "tokenizer_revision": manifest["tokenizer"]["tokenizer_revision"],
    "word_segmenter_version": manifest["word_segmentation"]["word_segmenter_version"],
    "segmenter_resource_hash_count": len(
        manifest["word_segmentation"]["word_segmenter_resource_hashes"]),
    "repository_commit": manifest["repository"]["resolved_commit"],
}, indent=2, sort_keys=True))
print()
print("Paste this line into configs/training/s1_mention_full_training.yaml (model:):")
print(f'  pinned_revision: "{PINNED_MODEL_REVISION}"')

## 7. Smoke checkpoint policy

    smoke checkpoint            = pipeline integrity artifact (ONE optimizer step)
    full-training initialization = approved pretrained ViHealthBERT base revision

`s1_mention_smoke_model.pt` must never initialize or resume full S1 training. The
full-training config enforces `initialize_from: pretrained_base`, and
`validate_resume_checkpoint()` rejects any payload whose `mode` is `SMOKE_ONLY`.

In [ ]:
from mednorm_vi.training.s1_full_training import (  # noqa: E402
    load_full_training_config,
    validate_resume_checkpoint,
)

# Full training consumes the artifact that was actually validated here.
full_config = load_full_training_config(
    REPO_DIR / "configs" / "training" / "s1_mention_full_training.yaml",
    pinned_revision=PINNED_MODEL_REVISION,
    smoke_artifact_dir=SMOKE_ARTIFACT_DIR,
)
assert full_config.initialize_from == "pretrained_base"
assert Path(full_config.output_dir) != SMOKE_ARTIFACT_DIR, (
    "full-training output must never overwrite the smoke artifact")
rejected = validate_resume_checkpoint({"mode": "SMOKE_ONLY"}, full_config)
assert rejected, "the smoke checkpoint must be rejected as a resume source"
print(json.dumps({
    "pinned_revision": full_config.pinned_revision,
    "initialize_from": full_config.initialize_from,
    "effective_batch_size": full_config.effective_batch_size,
    "full_training_output_dir": full_config.output_dir,
    "validated_smoke_artifact_dir": str(SMOKE_ARTIFACT_DIR),
    "validated_checkpoint_sha256": outcome.checkpoint_sha256,
    "smoke_checkpoint_rejected_as_resume": rejected[:1],
    "config_sha256": full_config.config_sha256,
}, indent=2, sort_keys=True))
print()
print("READY: run MedNorm_S1_Mention_Full_Training.ipynb with")
print(f'    SMOKE_ARTIFACT_DIR = "{SMOKE_ARTIFACT_DIR}"')
print(f'    EXPECTED_SMOKE_CHECKPOINT_SHA256 = "{outcome.checkpoint_sha256}"')
print("after pinning the revision in the full-training config.")